In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

# Support kernels launched from either the repository root or notebooks/.
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

train_dir = repo_root / "data" / "casting_data" / "casting_data" / "train"
defective_dir = train_dir / "def_front"
normal_dir = train_dir / "ok_front"

def first_image(directory: Path) -> Path:
    if not directory.is_dir():
        raise FileNotFoundError(f"Image folder not found: {directory}")
    images = sorted(path for path in directory.iterdir() if path.is_file())
    if not images:
        raise FileNotFoundError(f"No images found in: {directory}")
    return images[0]

defective_sample = first_image(defective_dir)
normal_sample = first_image(normal_dir)

img_def = cv2.imread(str(defective_sample), cv2.IMREAD_GRAYSCALE)
img_norm = cv2.imread(str(normal_sample), cv2.IMREAD_GRAYSCALE)
if img_def is None:
    raise ValueError(f"Could not read image: {defective_sample}")
if img_norm is None:
    raise ValueError(f"Could not read image: {normal_sample}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_def, cmap="gray")
axes[0].set_title("Defective Part (O.O.F. Target)")
axes[0].axis("off")

axes[1].imshow(img_norm, cmap="gray")
axes[1].set_title("Normal Part (OK)")
axes[1].axis("off")

plt.show()

In [ ]:
from pathlib import Path

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Use the extracted dataset layout: data/casting_data/casting_data/{train,test}.
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

dataset_root = repo_root / "data" / "casting_data" / "casting_data"
train_dir = dataset_root / "train"
test_dir = dataset_root / "test"

data_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Total training images: {len(train_dataset)}")
print(f"Total testing images: {len(test_dataset)}")
print(f"Classes detected by PyTorch: {train_dataset.classes}")

In [ ]:
from pathlib import Path

import cv2


def check_data_quality(directory):
    valid_count = 0
    corrupt_count = 0
    missing_folders = []

    for folder in ["def_front", "ok_front"]:
        folder_path = directory / folder
        if not folder_path.is_dir():
            missing_folders.append(folder_path)
            continue

        image_files = [path for path in folder_path.iterdir() if path.is_file()]
        if not image_files:
            missing_folders.append(folder_path)
            continue

        for filepath in image_files:
            if cv2.imread(str(filepath)) is None:
                corrupt_count += 1
            else:
                valid_count += 1

    return valid_count, corrupt_count, missing_folders


repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

dataset_root = repo_root / "data" / "casting_data" / "casting_data"
train_dir = dataset_root / "train"
test_dir = dataset_root / "test"

print("Running Data Quality Check...")
train_valid, train_corrupt, train_missing = check_data_quality(train_dir)
test_valid, test_corrupt, test_missing = check_data_quality(test_dir)

print(f"Training set: {train_valid} readable images, {train_corrupt} corrupt/unreadable files.")
print(f"Testing set: {test_valid} readable images, {test_corrupt} corrupt/unreadable files.")

for split_name, missing_folders in [("Training", train_missing), ("Testing", test_missing)]:
    for folder_path in missing_folders:
        print(f"{split_name} class folder missing or empty: {folder_path}")

if (
    train_valid > 0
    and test_valid > 0
    and train_corrupt == 0
    and test_corrupt == 0
    and not train_missing
    and not test_missing
):
    print("\nQuality Review Passed: all discovered train/test images are readable and expected class folders are present.")
else:
    print("\nQuality Review Failed: check the unreadable files and missing/empty class-folder messages above.")